In [ ]:
## fetching international commodities into one CSV

import requests
import pandas as pd
from time import sleep
import os

# Step 1: Fetch all international commodities
url_series = "https://fpma.fao.org/giews/v4/global/price_module/api/v1/FpmaSerieInternational/?limit=500"
series_data = requests.get(url_series).json()

all_rows = []  # ← collect everything here

# Step 2: Loop over all commodities
for item in series_data['results']:
    uuid = item['uuid']
    commodity_name = item.get('commodity_name', 'Unknown')
    country = item.get('market_name', 'Unknown')
    market = item.get('country_name', 'Unknown')
    price_type = item.get('price_type', 'Unknown')
    unit = item.get('measure_unit_label', 'Unknown')

    # Step 3: Fetch prices for this commodity
    url_price = (
        f"https://fpma.fao.org/giews/v4/global/price_module/api/v1/"
        f"FpmaSeriePrice/?uuid__in={uuid}&periodicity=monthly"
    )
    resp = requests.get(url_price).json()

    if resp['count'] == 0:
        continue

    datapoints = resp['results'][0]['datapoints']
    rows = []

    for dp in datapoints:
        price_usd = dp.get('price_value_dollar') or dp.get('price_value')
        if price_usd is None:
            continue

        rows.append({
            'date': dp['date'],
            'price_usd': price_usd,
            'commodity_name': commodity_name,
            'country': country,
            'market': market,
            'price_type': price_type,
            'unit': unit,
            'price_source': 'International'
        })

    if not rows:
        continue

    # ---- GAP FILLING ----
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])

    full_range = pd.date_range(
        start=df['date'].min(),
        end=df['date'].max(),
        freq='MS'
    )

    df = df.set_index('date').reindex(full_range)
    df['commodity_name'] = commodity_name
    df['country'] = country
    df['market'] = market
    df['price_type'] = price_type
    df['unit'] = unit
    df['price_source'] = 'International'

    df.index.name = "date"
    df = df.reset_index()
    df = df[['date', 'price_usd', 'commodity_name', 'country', 'market', 'price_type', 'unit', 'price_source']]

    all_rows.append(df)  # ← append this commodity's df to the list
    print(f"Fetched {commodity_name} ({len(df)} rows)")

    sleep(0.2)

# Step 4: Combine and save a single CSV
if all_rows:
    final_df = pd.concat(all_rows, ignore_index=True)
    final_df.to_csv("international_commodity.csv", index=False)
    print(f"\nSaved international_commodity.csv with {len(final_df)} total rows")
else:
    print("No data fetched.")
